# Mostrar cartas:

Función que muestra una carta por pantalla, se usa para debug

In [ ]:
from IPython.display import display, HTML

def display_card(url):
    # Ahora que tenemos la URL real, generamos el HTML
    html_code = f"""
    <div style="width: 300px; border-radius: 15px; overflow: hidden; box-shadow: 0 8px 16px rgba(0,0,0,0.3);">
        <img src="{url}" alt="Carta" style="width:100%; display: block;">
    </div>
    """
    display(HTML(html_code))

## Crear embeddings:

funcionalidad para crear embeddings apartir del texto y el nombre de una carta

In [ ]:
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

df = pd.read_csv("cards_final_with_xp.csv")

batch_size=32
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

try:
    texts = [f"{name} \n {text}" for name, text in zip(df["name"], df["text"])]
    query_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size).astype(np.float32)
finally:
    torch.cuda.empty_cache()

/home/yoxid/Escritorio/UPM/3º/2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 253.59it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
query_embeddings.shape

(5346, 384)

# Embeddings a Redis
Función que recibe un embedding como un array de numpy y devuelve un blob que se le puede pasar a redis

In [ ]:
import numpy as np

def to_blob(embedding: np.array) -> bytes:
    """
    Converts embedding to blob.
    :param embedding: embedding
    :return: blob
    """
    return embedding.astype(np.float32).tobytes()

# Iniciar conexión con redis

In [ ]:
import redis

r = redis.Redis(host='redis', port=6379)

In [ ]:
r.flushall()

In [ ]:
df

,code,name,text,type_code,traits,pack_code,illustrator,image_url,xp,faction_code
0,60401,Jacqueline Fine,[reaction] When an investigator at your locati...,investigator,Clairvoyant,jac,Aleksander Karcz,https://arkhamdb.com/bundles/cards/60401.jpg,0,mystic
1,60402,Arbiter of Fates,Jacqueline Fine deck only. [reaction] When you...,asset,Talent,jac,Pavel Kolomeyets,https://arkhamdb.com/bundles/cards/60402.jpg,0,mystic
2,60403,Dark Future,Revelation - Put Dark Future into play in your...,treachery,Omen|Endtimes,jac,Matt Bradbury,https://arkhamdb.com/bundles/cards/60403.jpg,0,neutral
3,60404,Nihilism,Revelation - Put Nihilism into play in your th...,treachery,Madness,jac,Sara Biddle,https://arkhamdb.com/bundles/cards/60404.jpg,0,neutral
4,60406,Scrying Mirror,Uses (4 secrets). [reaction] After a skill tes...,asset,Item|Charm,jac,Drazenka Kimpel,https://arkhamdb.com/bundles/cards/60406.jpg,0,mystic
...,...,...,...,...,...,...,...,...,...,...
5341,08129,Call for Backup,"One at a time, in any order, if you control a....",event,Favor|Synergy,eoep,Adam Schumpert,https://arkhamdb.com/bundles/cards/08129.jpg,2,neutral
5342,08130,Arm Injury,Revelation - Put Arm Injury into play in your ...,treachery,Injury,eoep,Robert Laskey,https://arkhamdb.com/bundles/cards/08130.jpg,0,neutral
5343,08131,Leg Injury,Revelation - Put Leg Injury into play in your ...,treachery,Injury,eoep,Robert Laskey,https://arkhamdb.com/bundles/cards/08131.jpg,0,neutral
5344,08132,Panic,Revelation - Put Panic into play in your threa...,treachery,Madness,eoep,Felicia Cano,https://arkhamdb.com/bundles/cards/08132.jpg,0,neutral


### Importar las cartas a redis

In [ ]:
import pandas as pd
import redis

# 1. Conexión a Redis
# Ajusta host, port y db según tu configuración
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

def import_csv_to_redis(file_path):
    # 2. Cargar el CSV con Pandas
    df = pd.read_csv(file_path)
    
    # Reemplazar valores NaN por strings vacíos (Redis no acepta nulos de Python)
    df = df.fillna("")

    print(f"Iniciando la carga de {len(df)} registros...")

    aux = 0
    # 3. Iterar sobre las filas y guardarlas en Redis
    for _, row in df.iterrows():
        # Creamos una clave única para cada carta, por ejemplo "card:60401"
        card_key = row['code'] 
        if aux == 0:
            print(f"Ejemplo de clave: card:{card_key}")
            print(f"Ejemplo de datos: {row.to_dict()}")
            aux += 1
        
        # Convertimos la fila a un diccionario
        card_data = row.to_dict()
        
        # Usamos HSET para guardar todos los campos en un Hash de Redis
        r.hset(f"card:{card_key}", mapping=card_data)

    print("¡Importación completada con éxito!")


import_csv_to_redis('cards_final_with_xp.csv')

Iniciando la carga de 5346 registros...
¡Importación completada con éxito!


### Crear las funciones de las acciones

In [ ]:
def card_exists(code):
    return r.exists(code) == 1

card_exists("60401")


False

In [ ]:
def ver_carta_code(code):
    if card_exists(code):
        card_data = r.hgetall(code)
        print(f"Datos de la carta {code}:")
        for key, value in card_data.items():
            print(f"  {key}: {value}")
        return
    print(f"No se encontró la carta con código {code}.")
    

ver_carta_code("60401")

No se encontró la carta con código 60401.


In [ ]:
def meter_carta(code, name, text, type_code, traits, pack_code, faction_code, xp, illustrator, image_url):
    card_data = {
        "code": code,
        "name": name,
        "text": text,
        "type_code": type_code,
        "traits": traits,
        "pack_code": pack_code,
        "faction_code": faction_code,
        "xp": xp,
        "illustrator": illustrator,
        "image_url": image_url
    }
    r.hset(code, mapping=card_data)

meter_carta("99999", "Carta de Prueba", "Este es un texto de prueba.", "type_test", "trait1, trait2", "pack_test", "faction_test", 5, "Test Illustrator", "http://example.com/image.jpg")

ver_carta_code("99999")

Datos de la carta 99999:
  code: 99999
  name: Carta de Prueba
  text: Este es un texto de prueba.
  type_code: type_test
  traits: trait1, trait2
  pack_code: pack_test
  faction_code: faction_test
  xp: 5
  illustrator: Test Illustrator
  image_url: http://example.com/image.jpg


In [ ]:
def eliminar_carta(code):
    if card_exists(code):
        r.delete(code)
        print(f"Carta con código {code} eliminada.")
    else:
        print(f"No se encontró la carta con código {code} para eliminar.")

eliminar_carta("99999")

ver_carta_code("99999")

Carta con código 99999 eliminada.
No se encontró la carta con código 99999.


# 2. INDICES

En  el  juego  hay  7  facciones,  5  que  son  clases  que  pueden  usar  los  jugadores  (mystic, 
survivor, guardian, seeker y rogue), la facción  “neutral” que es equipo común para todas 
las  clases  y  la  facción  “mythos”  que  son  los  enemigos.  Recientemente  se  han  añadido 
cartas  especiales  que  tienen  más  de  una  facción3.  Los  jugadores  están  interesados  en 
buscar todas las cartas que contengan varias facciones a la vez, así como, buscar todas las 
cartas que contengan al menos una de las facciones que indiquen. Por defecto devolvemos 
las cartas de 5 en 5. 

In [ ]:
import redis
from redis.commands.search.field import TextField, TagField, NumericField
from redis.commands.search.index_definition import IndexDefinition, IndexType

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# 1. Definimos el esquema (qué columnas queremos indexar)


try:
    r.ft("cartas:").create_index(
        schema = (
            NumericField("code"),           # NUMERIC para búsquedas exactas por código
            TextField("name"),  # TEXT para búsquedas de palabras (el nombre pesa más 
            TextField("text"),              # TEXT para el cuerpo de la carta
            TextField("type_code"),         # TEXT para el tipo de carta
            TagField("traits", separator="|"), # TAG indicando que los rasgos se separan por '|'
            TagField("pack_code"),          # TAG para el paquete al que pertenece la carta
            TagField("faction_code", separator="|"),       # TAG para la facción a la que pertenece la carta
            NumericField("xp"),          # NUMERIC para filtros de rango (ej: xp > 2)
            TextField("illustrator"),       # TEXT para el ilustrador de la carta
            TextField("image_url")        # TEXT para la URL de la imagen de la carta  
        ),
        
        definition=IndexDefinition(
            prefix=["card:"],       # SOLO indexará llaves que empiecen por "card:"
            index_type=IndexType.HASH
        )
    )
    print("✅ Índice creado con éxito")
except Exception as e:
    print(f"❌ Error o el índice ya existe: {e}")

✅ Índice creado con éxito


In [ ]:
aaaa

NameError: name 'aaaa' is not defined

In [ ]:
r.flushall()

True